# Three Jacobians You Should Never Build

**Topic:** Jacobian-vector products &nbsp;·&nbsp; **Difficulty:** Hard

A Jacobian for a layer with 2,000 inputs and 2,000 outputs has four million
entries. You almost never want it. What you want is its action on a vector, and
for the three layers here, that action has a closed form that costs no more than
the forward pass.

Build each Jacobian-vector product directly. The softmax one is the interesting
case: its Jacobian is dense, but the product collapses to two cheap terms.

## The data (created in the next cell)

- `z` `(2000,)` and `v` `(2000,)`: an input and a tangent direction.
- `W` `(2000, 2000)` for the linear case.
- Three maps: elementwise `tanh`, the linear map `z -> W @ z`, and `softmax(z)`.

## Your task

With **NumPy only**, and **without ever forming a (2000, 2000) Jacobian**:

1. `jvp_tanh`: the JVP of elementwise `tanh` at `z` applied to `v`.
2. `jvp_linear`: the JVP of `z -> W @ z` applied to `v`.
3. `jvp_softmax`: the JVP of `softmax` at `z` applied to `v`. The Jacobian is
   `diag(p) - p p^T`, so the product is `p * v - p * (p @ v)`.
4. Set `answers` to a dict with `"jvp_tanh"`, `"jvp_linear"`, `"jvp_softmax"`
   (each `(2000,)`).

## Evaluation

Each is checked against reference values (its sum and one fixed random projection)
to a relative tolerance of 1e-6, and the `tanh` case is additionally checked against
a central-difference directional derivative.

In [ ]:
import numpy as np

rng = np.random.default_rng(83)
m = 2000
z = rng.normal(0, 1.0, m)
v = rng.normal(0, 1.0, m)
W = rng.normal(0, 1.0 / np.sqrt(m), (m, m))
print(f"z {z.shape}, W {W.shape}. The dense Jacobian would be {m * m:,} entries")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("jvp_tanh", [2000], 44.62200660769072, -6.703199253337902, 1e-06),
    ("jvp_linear", [2000], -72.80768144522472, 64.40206992441692, 1e-06),
    ("jvp_softmax", [2000], 2.0816681711721685e-17, 0.0008970261782936997, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

_h = 1e-6
_fd_tanh = (np.tanh(z + _h * v) - np.tanh(z - _h * v)) / (2 * _h)
_rel = float(np.linalg.norm(np.asarray(answers.get("jvp_tanh", 0.0), dtype=float) - _fd_tanh)
             / np.linalg.norm(_fd_tanh))
print(f"  {'matches finite diff':<24} {'ok' if _rel < 1e-5 else 'wrong'}  (rel err = {_rel:.2e})")
_ok = _ok and _rel < 1e-5

print("=" * 56)
if _ok:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

For an elementwise map the Jacobian is diagonal, so the JVP is just a product.
For a linear map the Jacobian is `W` itself. For softmax, `diag(p) - p p^T`
applied to `v` expands to `p * v - p * (p . v)`, which is one elementwise product and one
dot product, never the outer product. That structure is exactly why reverse-mode
autodiff can afford softmax layers at vocabulary scale.

```python
import numpy as np

t = np.tanh(z)
jvp_tanh = (1.0 - t ** 2) * v

jvp_linear = W @ v

zs = z - z.max()
e = np.exp(zs)
p = e / e.sum()
jvp_softmax = p * v - p * float(p @ v)

answers = {
    "jvp_tanh": jvp_tanh,
    "jvp_linear": jvp_linear,
    "jvp_softmax": jvp_softmax,
}
```